<a href="https://colab.research.google.com/github/TaherBenAfia/Fly2/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

 1) # **myth#7 : "Fresh Content Always Outperforms."**
    this is a very controversial take because checking the correlation between old content and up trend ( target ) , you would find a strong correlation, from what I've seen in the paper the impressions_90  are more for the recent content but 30d-vs-prev-30d impression (Trend direction) shows more true positives for the older content


2) # **ML appendix : "Average Position is the #1 predictor"**

    I think this is pretty solid and logical to say , even if the feature importance report never happened , you would assume that better pages mean pages higher on the position tier list, but it does leave bunch of factors behind as the positions themselves are a target not yet a feature/variable of predicition ( if we're talking web visibility )

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [2]:
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, KFold
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
y = df["is_declining_label"].values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

df["impressions_earliest_slice"] = (df["impressions_90d"] - df["impressions_last_30d"] - df["impressions_prev_30d"]).clip(lower=0)
df["log_impressions_earliest_slice"] = np.log1p(df["impressions_earliest_slice"])
df.loc[df["avg_position"] == 0, "avg_position"] = np.nan

numeric_features = ["search_volume","competition","cpc","word_count","char_count",
    "log_impressions_earliest_slice","content_age_days","days_since_last_update",
    "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
categorical_features = ["competition_level","content_type","main_intent","age_tier",
    "freshness_tier","word_count_tier","impression_tier","position_tier"]

X_num = df[numeric_features].fillna(0)
X_cat = pd.get_dummies(df[categorical_features].fillna("unknown"))
X = pd.concat([X_num, X_cat], axis=1)

print("=== BEFORE: naive row-level KFold (ignores client grouping) ===")
kf = KFold(n_splits=5, shuffle=True, random_state=42)
naive_aucs, naive_p50 = [], []
for train_idx, test_idx in kf.split(X):
    rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(X.iloc[train_idx], y[train_idx])
    proba = rf.predict_proba(X.iloc[test_idx])[:,1]
    naive_aucs.append(roc_auc_score(y[test_idx], proba))
    naive_p50.append(precision_at_k(proba, y[test_idx], 50))
print("naive mean AUC:", np.mean(naive_aucs), "std:", np.std(naive_aucs))
print("naive mean P@50:", np.mean(naive_p50), "std:", np.std(naive_p50))

print("\n=== AFTER: honest GroupKFold on client_id ===")
gkf = GroupKFold(n_splits=5)
grouped_aucs, grouped_p50 = [], []
for train_idx, test_idx in gkf.split(X, y, groups=df["client_id"]):
    rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
    rf.fit(X.iloc[train_idx], y[train_idx])
    proba = rf.predict_proba(X.iloc[test_idx])[:,1]
    grouped_aucs.append(roc_auc_score(y[test_idx], proba))
    grouped_p50.append(precision_at_k(proba, y[test_idx], 50))
print("grouped mean AUC:", np.mean(grouped_aucs), "std:", np.std(grouped_aucs))
print("grouped mean P@50:", np.mean(grouped_p50), "std:", np.std(grouped_p50))

=== BEFORE: naive row-level KFold (ignores client grouping) ===
naive mean AUC: 0.7510821600313585 std: 0.005029717308993513
naive mean P@50: 0.9399999999999998 std: 0.012649110640673493

=== AFTER: honest GroupKFold on client_id ===
grouped mean AUC: 0.6499175760146246 std: 0.0538622992368137
grouped mean P@50: 0.732 std: 0.18004443895883038


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
import pandas as pd, numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
y = df["trend_direction"].str.lower().eq("down").astype(int)

df["impressions_earliest_slice"] = (df["impressions_90d"] - df["impressions_last_30d"] - df["impressions_prev_30d"]).clip(lower=0)
df["log_impressions_earliest_slice"] = np.log1p(df["impressions_earliest_slice"])

final_numeric_features = ["search_volume","competition","cpc","word_count","char_count",
    "log_impressions_earliest_slice","content_age_days","days_since_last_update",
    "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]

controls = ["trend_pct","impressions_90d","impressions_last_30d","impressions_prev_30d"]

for col in final_numeric_features + controls:
    x = df[[col]].fillna(0)
    t = DecisionTreeClassifier(max_depth=1, random_state=42).fit(x, y)
    auc = roc_auc_score(y, t.predict_proba(x)[:,1])
    flag = "LEAK" if auc > 0.75 else ("watch" if auc > 0.65 else "ok")
    print(f"{col:28s} {auc:.3f} {flag}")

search_volume                0.527 ok
competition                  0.506 ok
cpc                          0.501 ok
word_count                   0.566 ok
char_count                   0.563 ok
log_impressions_earliest_slice 0.593 ok
content_age_days             0.586 ok
days_since_last_update       0.547 ok
ctr                          0.540 ok
avg_position                 0.545 ok
engagement_rate              0.505 ok
scroll_rate                  0.513 ok
ai_traffic_pct               0.502 ok
trend_pct                    1.000 LEAK
impressions_90d              0.581 ok
impressions_last_30d         0.532 ok
impressions_prev_30d         0.623 ok


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

# I don't think I have a bold take here other than the old age content being better than fresh on paper. and I described the measures in the first notebook.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.